# Pengujian Hipotesis

Perkiraan waktu: **30** menit

Data pelanggan memegang peranan penting bagi hampir semua bisnis. Bayangkan kamu bekerja sebagai analis asuransi yang harus menilai risiko calon nasabah. Sebagian tugasnya adalah memeriksa atribut seperti usia, jenis kelamin, BMI, status merokok, dan wilayah tempat tinggal, lalu memakainya untuk memutuskan apakah klaim disetujui atau ditolak.

Notebook ini memandu langkah-langkah pengujian hipotesis untuk menilai pengaruh suatu atribut atau kombinasi atribut terhadap klaim asuransi. Pembahasan berhenti pada tahap pengambilan keputusan, sebelum masuk ke pembangunan model prediksi dan klasifikasi.

## Tujuan

Setelah menyelesaikan latihan ini, kamu dapat:

- Memahami elemen-elemen pengujian hipotesis
  - Memilih statistik sampel
  - Menentukan hipotesis
  - Menetapkan kriteria keputusan
  - Menilai dan menafsirkan hasil


## Persiapan

Pustaka yang dipakai:

- `pandas` dan `numpy` untuk mengelola serta menghitung data.
- `seaborn` dan `matplotlib` untuk visualisasi.
- `scipy.stats` untuk uji t dan chi-square.
- `statsmodels` untuk tabel ANOVA.


In [ ]:
# Imports
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings("ignore")


## Membaca dan memahami data

Data yang dipakai adalah `insurance.csv`, berisi usia, jenis kelamin, BMI, jumlah anak, status merokok, wilayah tempat tinggal, dan biaya asuransi setiap nasabah.


In [ ]:
# Unduh data sekali saja, lalu muat ke DataFrame
data_path = Path("insurance.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/insurance.csv"

if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)

insurance_data = pd.read_csv(data_path)
insurance_data.head()


Tipe data setiap kolom diperiksa dengan `info()`.


In [ ]:
insurance_data.info()


Statistik ringkas kolom numerik dilihat dengan `describe()`.


In [ ]:
insurance_data.describe()


Variabel respons pada latihan ini adalah `charges`, sedangkan prediktornya `age`, `sex`, `bmi`, `children`, `smoker`, dan `region`. Yang diuji adalah pengaruh prediktor terhadap besarnya biaya asuransi.

## Langkah pengujian hipotesis

## Contoh 1

Contoh pertama menguji apakah BMI perempuan berbeda dari BMI laki-laki.

### 1. Memilih statistik sampel

Langkah pertama adalah menentukan statistik uji. Misalkan $\mu_1$ adalah rata-rata BMI populasi laki-laki dan $\mu_2$ rata-rata BMI populasi perempuan. Kedua nilai ini dibandingkan secara statistik.

### 2. Menentukan hipotesis nol dan alternatif

Hipotesis nol menganggap perbedaan pengamatan terjadi karena faktor acak. Hipotesis alternatif menganggap perbedaannya nyata.

$H_0: \mu_1 - \mu_2 = 0$ — tidak ada perbedaan BMI laki-laki dan perempuan.

$H_A: \mu_1 - \mu_2 \neq 0$ — ada perbedaan BMI laki-laki dan perempuan.

Tanda sama dengan pada hipotesis nol menunjukkan uji dua arah.

### 3. Menetapkan kriteria keputusan

Tingkat signifikansi ditetapkan lebih dulu, misalnya 5%, 1%, atau 0.5%. Di sini dipilih $\alpha = 0.05$. Karena ujinya dua arah, alpha dibagi dua menjadi 0.025. Bila p-value lebih kecil dari alpha, hipotesis nol ditolak.

Statistik yang bisa dipakai antara lain t-test, z-score, f-score, atau chi-square:

- **t-test** membandingkan rata-rata satu populasi dengan nilai acuan, atau rata-rata dua populasi saat simpangan baku populasi tidak diketahui dan sampel terbatas.
- **z-test** dipakai untuk sampel besar atau saat simpangan baku populasi diketahui, termasuk untuk membandingkan proporsi.
- **f-test** membandingkan varians dua populasi dan menjadi dasar ANOVA.
- **chi-square** menguji apakah ada perbedaan signifikan antara frekuensi harapan dan frekuensi pengamatan pada tabel kontingensi.

### 4. Menilai dan menafsirkan hasil

Observasi perempuan dan laki-laki dipisahkan lebih dulu.


In [ ]:
female_data = insurance_data.loc[insurance_data["sex"] == "female"]
male_data = insurance_data.loc[insurance_data["sex"] == "male"]

female_bmi = female_data["bmi"]
male_bmi = male_data["bmi"]


Sebaran BMI kedua kelompok digambarkan sebagai kurva kepadatan.


In [ ]:
sns.histplot(female_bmi, color="green", kde=True, stat="density", element="step", fill=False)
sns.histplot(male_bmi, color="red", kde=True, stat="density", element="step", fill=False)
plt.xlabel("BMI")
plt.show()


Kedua kurva terlihat sangat mirip, jadi perbedaan rata-ratanya kemungkinan kecil. Rata-rata masing-masing kelompok dihitung untuk memastikan.


In [ ]:
print(f"Rata-rata BMI perempuan: {female_bmi.mean():.4f}")
print(f"Rata-rata BMI laki-laki: {male_bmi.mean():.4f}")


Nilai t dan p dihitung dengan `ttest_ind()`. Sebelum itu, satu fungsi bantu disiapkan agar kesimpulan setiap uji bisa dicetak dengan format yang sama.


In [ ]:
def report_test(p_value, alpha, reject_text, fail_text):
    """Cetak kesimpulan uji hipotesis berdasarkan perbandingan p-value dan alpha."""
    if p_value < alpha:
        print(f"p-value {p_value:.6g} < alpha {alpha}: tolak H0.")
        print(reject_text)
    else:
        print(f"p-value {p_value:.6g} >= alpha {alpha}: gagal menolak H0.")
        print(fail_text)

alpha = 0.05
t_value_bmi, p_value_bmi = stats.ttest_ind(male_bmi, female_bmi)
print(f"t = {t_value_bmi:.4f}, p = {p_value_bmi:.4f}")


In [ ]:
report_test(
    p_value_bmi,
    alpha,
    "Ada perbedaan signifikan antara BMI perempuan dan laki-laki.",
    "Tidak ada perbedaan signifikan antara BMI perempuan dan laki-laki.",
)


p-value sekitar 0.09, di atas alpha. Hipotesis nol tidak ditolak, sehingga BMI perempuan dan laki-laki tidak berbeda secara signifikan.

## Contoh 2

Contoh kedua menguji apakah klaim medis perokok lebih besar daripada bukan perokok. Rata-rata biaya asuransi kedua kelompok dihitung lebih dulu.


In [ ]:
smoker_charges = insurance_data.loc[insurance_data["smoker"] == "yes", "charges"]
nonsmoker_charges = insurance_data.loc[insurance_data["smoker"] == "no", "charges"]

print(f"Rata-rata biaya perokok: {smoker_charges.mean():.2f}")
print(f"Rata-rata biaya bukan perokok: {nonsmoker_charges.mean():.2f}")


## Latihan 1

Hitung rata-rata biaya asuransi kelompok bukan perokok, lalu bandingkan dengan rata-rata perokok.


In [ ]:
nonsmoker_mean = nonsmoker_charges.mean()
smoker_mean = smoker_charges.mean()
print(f"Selisih rata-rata: {smoker_mean - nonsmoker_mean:.2f}")


Hipotesis untuk contoh ini:

$H_0: \mu_1 \leq \mu_2$ — rata-rata biaya perokok tidak lebih besar daripada bukan perokok.

$H_A: \mu_1 > \mu_2$ — rata-rata biaya perokok lebih besar daripada bukan perokok.

Tanda `>` pada hipotesis alternatif menunjukkan uji satu arah (ekor kanan). Bila nilai t jatuh di area kanan kurva distribusi, hipotesis nol ditolak.

Sebaran biaya kedua kelompok digambarkan dengan boxplot.


In [ ]:
sns.boxplot(x="charges", y="smoker", data=insurance_data)
plt.title("Perokok vs Biaya Asuransi")
plt.show()


Nilai t dan p dihitung dengan `ttest_ind()`. Karena ujinya satu arah, p dua arah dibagi dua.


In [ ]:
t_value_charges, p_value_charges = stats.ttest_ind(smoker_charges, nonsmoker_charges)
p_value_one_tail = p_value_charges / 2
print(f"t = {t_value_charges:.4f}, p dua arah = {p_value_charges:.3e}, p satu arah = {p_value_one_tail:.3e}")


## Latihan 2

Cetak kesimpulan berdasarkan statistik di atas. Apakah hipotesis nol ditolak?


In [ ]:
report_test(
    p_value_one_tail,
    alpha,
    "Rata-rata biaya perokok lebih besar secara signifikan daripada bukan perokok.",
    "Rata-rata biaya perokok tidak terbukti lebih besar daripada bukan perokok.",
)


p satu arah jauh di bawah alpha, sehingga hipotesis nol ditolak. Biaya asuransi perokok terbukti lebih tinggi.

## Contoh 3

Contoh ketiga membandingkan BMI perempuan tanpa anak, dengan satu anak, dan dengan dua anak.

$H_0: \mu_1 = \mu_2 = \mu_3$ — rata-rata BMI ketiga kelompok sama.

$H_A$: minimal satu rata-rata BMI berbeda.

Pengujian memakai one-way ANOVA dengan statistik f-score dan $\alpha = 0.05$. Data perempuan dengan 0, 1, dan 2 anak disaring lebih dulu.


In [ ]:
female_small_family = female_data.loc[female_data["children"] <= 2]
female_small_family.groupby("children")["bmi"].mean()


## Latihan 3

Gambarkan boxplot untuk melihat perbedaan BMI antar ketiga kelompok.


In [ ]:
sns.boxplot(x="children", y="bmi", data=female_small_family)
plt.grid()
plt.show()


Tabel ANOVA disusun memakai model OLS, yaitu estimasi parameter dengan metode kuadrat terkecil.


In [ ]:
anova_model = ols("bmi ~ C(children)", female_small_family).fit()
anova_table = anova_lm(anova_model)
anova_table


p-value sekitar 0.716, jauh di atas alpha 0.05. Hipotesis nol tidak ditolak, sehingga rata-rata BMI perempuan tanpa anak, satu anak, dan dua anak dianggap sama.

## Contoh 4

Contoh keempat menguji apakah proporsi perokok berbeda secara signifikan antar wilayah.

$H_0$: proporsi perokok tidak berbeda antar wilayah.

$H_A$: proporsi perokok berbeda antar wilayah.

Karena kedua variabel bersifat kategorikal, pengujian memakai chi-square. Tabel kontingensi dibentuk dengan `crosstab()`.


In [ ]:
contingency_table = pd.crosstab(insurance_data["region"], insurance_data["smoker"])
contingency_table


Sebaran perokok dan bukan perokok pada setiap wilayah digambarkan sebagai diagram batang.


In [ ]:
contingency_table.plot(kind="bar")
plt.show()


`chi2_contingency()` menghasilkan statistik chi-square, p-value, derajat kebebasan, dan frekuensi harapan.


In [ ]:
chi2_stat, chi2_p, dof, expected_freq = chi2_contingency(contingency_table, correction=False)
print(f"chi-square: {chi2_stat:.4f}, p-value: {chi2_p:.4f}, dof: {dof}")
print(expected_freq)


## Latihan 4

Cetak kesimpulan berdasarkan hasil chi-square di atas.


In [ ]:
report_test(
    chi2_p,
    alpha,
    "Proporsi perokok berbeda secara signifikan antar wilayah.",
    "Proporsi perokok tidak berbeda secara signifikan antar wilayah.",
)


p-value sekitar 0.062, sedikit di atas alpha 0.05. Hipotesis nol tidak ditolak, sehingga belum ada bukti cukup bahwa proporsi perokok berbeda antar wilayah. Perlu dicatat, selisihnya tipis sehingga penambahan data bisa mengubah kesimpulan.
